# 24 — Hybrid Stacking Ensemble

**Architecture:** Meta-learner (Logistic Regression) trained on the probability outputs of all 22 base models.

## What is Stacking?

Stacking (stacked generalisation) is a two-layer ensemble strategy:

```
Raw Text
    ├─► TF-IDF (50K)  →  18 Classical ML Models  ─┐
    └─► Tokenizer     →   4 Deep Learning Models  ─┤
                                                   │
                          22-feature vector        │
                         [prob_real per model] ◄───┘
                                   │
                    Logistic Regression Meta-Learner
                                   │
                           Fake / Real + Confidence
```

The meta-learner **learns which base models to trust** for which patterns — it down-weights weak models and up-weights reliable ones automatically.

## 1. Setup

In [ ]:
import os, sys, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import setup_paths, plot_confusion_matrix, plot_roc_curve, plot_pr_curve
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, roc_curve,
    precision_recall_curve, average_precision_score
)
from sklearn.linear_model import LogisticRegression

BASE_DIR, FIGURES_DIR = setup_paths()
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
CLASSICAL_DIR = os.path.join(MODELS_DIR, 'classical')
DL_DIR        = os.path.join(MODELS_DIR, 'deep_learning')
HYBRID_DIR    = os.path.join(MODELS_DIR, 'hybrid')
os.makedirs(HYBRID_DIR, exist_ok=True)

COLORS = {'Classical ML': '#3498db', 'Deep Learning': '#e74c3c', 'Hybrid': '#2ecc71'}
print('Ready.')

## 2. Load Data

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 3. Load All Base Models

> **Prerequisite:** Run `export_models.py` first to generate all `.joblib` and `.keras` files.

In [ ]:
def get_prob(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        s = model.decision_function(X)
        s = s if s.ndim == 1 else s[:, 1]
        return (s - s.min()) / (s.max() - s.min() + 1e-10)
    return model.predict(X).astype(float)

# Load TF-IDF and dense pipeline
tfidf = joblib.load(os.path.join(CLASSICAL_DIR, 'tfidf.joblib'))
dense = joblib.load(os.path.join(CLASSICAL_DIR, 'dense_pipeline.joblib'))

# (registry_key, feature_type)
classical_specs = [
    ('logistic_regression', 'tfidf'), ('multinomial_nb',    'tfidf'),
    ('bernoulli_nb',        'tfidf'), ('complement_nb',     'tfidf'),
    ('linear_svm',          'tfidf'), ('decision_tree',     'tfidf'),
    ('random_forest',       'tfidf'), ('extra_trees',       'tfidf'),
    ('adaboost',            'tfidf'), ('ridge_classifier',  'tfidf'),
    ('sgd_classifier',      'tfidf'), ('passive_aggressive','tfidf'),
    ('perceptron',          'tfidf'), ('mlp_classifier',    'tfidf'),
    ('knn',                 'tfidf'), ('gradient_boosting', 'dense'),
    ('lightgbm',            'tfidf'), ('catboost',          'dense'),
]

classical_models = []
for name, feat in classical_specs:
    path = os.path.join(CLASSICAL_DIR, f'{name}.joblib')
    if os.path.exists(path):
        classical_models.append((name, feat, joblib.load(path)))
        print(f'  Loaded: {name}')
    else:
        print(f'  Skipped (not found): {name}')

print(f'\nClassical models loaded: {len(classical_models)}')

In [ ]:
dl_models = []
tokenizer = None
max_len = 200

try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import tokenizer_from_json
    from tensorflow.keras.preprocessing.sequence import pad_sequences

    with open(os.path.join(DL_DIR, 'tokenizer.json')) as f:
        tokenizer = tokenizer_from_json(f.read())
    with open(os.path.join(DL_DIR, 'config.json')) as f:
        dl_config = json.load(f)
    max_len = dl_config.get('max_len', 200)

    for dl_name in ['lstm', 'gru', 'cnn', 'cnn_lstm']:
        p = os.path.join(DL_DIR, f'{dl_name}.keras')
        if os.path.exists(p):
            dl_models.append((dl_name, tf.keras.models.load_model(p)))
            print(f'  Loaded DL: {dl_name}')
    print(f'\nDL models loaded: {len(dl_models)}')
except ImportError:
    print('TensorFlow not available — stacking with classical models only.')

total_base = len(classical_models) + len(dl_models)
print(f'\nTotal base models: {total_base}')

## 4. Generate Meta-Features

We hold out **20% of the training data** as the meta-training set. The base models generate probability scores on this holdout — these become the 22 input features for the meta-learner.

The final evaluation is done on `X_test`, which neither the base models nor the meta-learner have seen.

In [ ]:
# Hold out 20% of training data for meta-learner training
np.random.seed(42)
meta_idx = np.random.choice(len(X_train), size=int(0.20 * len(X_train)), replace=False)
X_meta = X_train[meta_idx]
y_meta = y_train[meta_idx]
print(f'Meta-train size : {len(X_meta):,}  (Fake={( y_meta==0).sum():,}  Real={(y_meta==1).sum():,})')
print(f'Test size       : {len(X_test):,}')

# Pre-transform features once for efficiency
print('\nVectorizing meta-train set...')
X_meta_tfidf = tfidf.transform(X_meta)
X_meta_dense = dense['svd'].transform(dense['tfidf'].transform(X_meta))

print('Vectorizing test set...')
X_test_tfidf = tfidf.transform(X_test)
X_test_dense = dense['svd'].transform(dense['tfidf'].transform(X_test))

if dl_models and tokenizer is not None:
    print('Tokenizing for DL models...')
    X_meta_seq = pad_sequences(tokenizer.texts_to_sequences(X_meta),
                               maxlen=max_len, padding='post', truncating='post')
    X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test),
                               maxlen=max_len, padding='post', truncating='post')
else:
    X_meta_seq = X_test_seq = None

print('Done.')

In [ ]:
base_model_names = []
meta_train_cols  = []
meta_test_cols   = []

print('Generating base model probabilities...')
for name, feat, model in classical_models:
    Xm = X_meta_tfidf if feat == 'tfidf' else X_meta_dense
    Xt = X_test_tfidf  if feat == 'tfidf' else X_test_dense
    meta_train_cols.append(get_prob(model, Xm))
    meta_test_cols.append(get_prob(model, Xt))
    base_model_names.append(name)
    print(f'  {name}: done')

if dl_models and X_meta_seq is not None:
    for dl_name, dl_model in dl_models:
        meta_train_cols.append(dl_model.predict(X_meta_seq, verbose=0).flatten())
        meta_test_cols.append(dl_model.predict(X_test_seq, verbose=0).flatten())
        base_model_names.append(dl_name)
        print(f'  {dl_name}: done')

X_meta_stack = np.column_stack(meta_train_cols)  # (n_meta, n_models)
X_test_stack = np.column_stack(meta_test_cols)   # (n_test,  n_models)

print(f'\nMeta-feature matrix shape (train): {X_meta_stack.shape}')
print(f'Meta-feature matrix shape (test) : {X_test_stack.shape}')
print(f'Features (columns) = one prob_real per base model')

## 5. Base Model Probability Correlation

If all base models were identical, stacking would add nothing. A healthy correlation heatmap shows some disagreement — these are the gaps the meta-learner exploits.

In [ ]:
corr_df = pd.DataFrame(X_test_stack, columns=base_model_names)
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True, ax=ax,
            linewidths=0.4, cbar_kws={'shrink': 0.8}, annot_kws={'size': 7})
ax.set_title('Base Model Prediction Correlation (test set)', fontsize=14, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'hybrid_correlation_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Print pairs with lowest correlation (most complementary)
corr_vals = corr.where(mask == False).stack().sort_values()
print('Most complementary model pairs (lowest correlation):')
print(corr_vals.head(8).to_string())

## 6. Train Meta-Learner

In [ ]:
MODEL_NAME = 'Hybrid Stacking (LR)'

t0 = time.time()
meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_lr.fit(X_meta_stack, y_meta)
train_time = time.time() - t0

print(f'Meta-learner trained in {train_time:.2f}s')
print(f'Intercept : {meta_lr.intercept_[0]:.4f}')
print(f'N features: {meta_lr.n_features_in_} (one per base model)')

## 7. Evaluation

In [ ]:
y_pred  = meta_lr.predict(X_test_stack)
y_prob  = meta_lr.predict_proba(X_test_stack)[:, 1]

metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'Recall'   : recall_score(y_test, y_pred, average='weighted', zero_division=0),
    'F1 Score' : f1_score(y_test, y_pred, average='weighted', zero_division=0),
    'AUC-ROC'  : roc_auc_score(y_test, y_prob),
    'Train Time': round(train_time, 2),
}

print(f"\n{'='*47}")
print(f'  {MODEL_NAME} — Evaluation Results')
print(f"{'='*47}")
for k, v in metrics.items():
    if k == 'Train Time':
        print(f'  {k:<18}: {v:.2f}s')
    else:
        print(f'  {k:<18}: {v:.4f}')
print(f"{'='*47}\n")
print(classification_report(y_test, y_pred, target_names=['Fake (0)', 'Real (1)']))

## 8. Confusion Matrix

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

## 9. ROC Curve

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## 10. Precision-Recall Curve

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## 11. Meta-Learner Weights — Which Models Are Trusted Most?

The logistic regression coefficients tell us how much the meta-learner relies on each base model's prediction. A high positive coefficient means that model is strongly trusted for predicting **real** news.

In [ ]:
coefs = meta_lr.coef_[0]
coef_df = pd.DataFrame({'model': base_model_names, 'weight': coefs})
coef_df['abs_weight'] = coef_df['weight'].abs()
coef_df = coef_df.sort_values('weight', ascending=True)

# Assign category colours
dl_set = {name for name, _ in dl_models} if dl_models else set()
bar_colors = ['#e74c3c' if m in dl_set else '#3498db' for m in coef_df['model']]

fig, ax = plt.subplots(figsize=(10, max(6, len(coef_df) * 0.42)))
bars = ax.barh(coef_df['model'], coef_df['weight'], color=bar_colors,
               edgecolor='white', height=0.7)
ax.axvline(0, color='black', lw=0.8)
for bar, val in zip(bars, coef_df['weight']):
    xpos = val + (0.005 if val >= 0 else -0.005)
    ha   = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', ha=ha, fontsize=8)
patches = [
    mpatches.Patch(color='#3498db', label='Classical ML'),
    mpatches.Patch(color='#e74c3c', label='Deep Learning'),
]
ax.legend(handles=patches, fontsize=10)
ax.set_xlabel('Meta-Learner Coefficient (contribution to Real prediction)')
ax.set_title('Hybrid Stacking — Base Model Weights')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'hybrid_model_weights.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most influential models:')
print(coef_df.sort_values('abs_weight', ascending=False).head(5)[['model','weight']].to_string(index=False))

## 12. Threshold Analysis

In [ ]:
thresholds = np.linspace(0.1, 0.9, 81)
accs, f1s, precs, recs = [], [], [], []

for t in thresholds:
    yp = (y_prob >= t).astype(int)
    accs.append(accuracy_score(y_test, yp))
    f1s.append(f1_score(y_test, yp, average='weighted', zero_division=0))
    precs.append(precision_score(y_test, yp, average='weighted', zero_division=0))
    recs.append(recall_score(y_test, yp, average='weighted', zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, accs,  lw=2, label='Accuracy')
ax.plot(thresholds, f1s,   lw=2, label='F1 (weighted)')
ax.plot(thresholds, precs, lw=2, label='Precision')
ax.plot(thresholds, recs,  lw=2, label='Recall')
ax.axvline(0.5, color='grey', linestyle='--', lw=1.2, label='Default threshold (0.5)')
best_t = thresholds[np.argmax(f1s)]
ax.axvline(best_t, color='green', linestyle=':', lw=1.5,
           label=f'Best F1 threshold ({best_t:.2f})')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Hybrid Stacking — Threshold vs Metrics')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'hybrid_threshold_analysis.png'),
            dpi=150, bbox_inches='tight')
plt.show()

best_f1_idx = np.argmax(f1s)
print(f'Best F1 threshold : {thresholds[best_f1_idx]:.2f}')
print(f'  Accuracy  : {accs[best_f1_idx]:.4f}')
print(f'  F1        : {f1s[best_f1_idx]:.4f}')
print(f'  Precision : {precs[best_f1_idx]:.4f}')
print(f'  Recall    : {recs[best_f1_idx]:.4f}')

## 13. Compare Hybrid vs All Individual Models

In [ ]:
results_csv = os.path.join(BASE_DIR, 'results', 'all_results.csv')
if os.path.exists(results_csv):
    df_res = pd.read_csv(results_csv).dropna(subset=['accuracy'])
    # Append hybrid result
    hybrid_row = pd.DataFrame([{
        'model'   : MODEL_NAME,
        'category': 'Hybrid',
        'accuracy': metrics['Accuracy'],
        'f1'      : metrics['F1 Score'],
        'auc'     : metrics['AUC-ROC'],
        'train_time': metrics['Train Time'],
    }])
    df_all = pd.concat([df_res, hybrid_row], ignore_index=True)
    df_all = df_all.sort_values('accuracy', ascending=False).reset_index(drop=True)

    print('Top 10 models by accuracy:')
    print(df_all[['model','category','accuracy','f1','auc']].head(10).to_string(index=False))

    # Find hybrid rank
    rank = df_all[df_all['model'] == MODEL_NAME].index[0] + 1
    print(f'\nHybrid rank: #{rank} out of {len(df_all)} models')

    # Bar chart: top 15 + hybrid
    top15 = df_all.head(15).sort_values('accuracy', ascending=True)
    bar_c = [COLORS.get(c, '#95a5a6') for c in top15['category']]

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(top15['model'], top15['accuracy'], color=bar_c,
                   edgecolor='white', height=0.7)
    for bar, val in zip(bars, top15['accuracy']):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    patches = [mpatches.Patch(color=v, label=k) for k, v in COLORS.items()]
    ax.legend(handles=patches, fontsize=10)
    ax.set_xlabel('Test Accuracy')
    ax.set_title('Top 15 Models — Accuracy Comparison (Hybrid Highlighted)')
    ax.set_xlim(0, 1.1)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'hybrid_comparison_top15.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('results/all_results.csv not found — run export_models.py first.')

## 14. Error Analysis

Where does the hybrid correct mistakes made by individual models?

In [ ]:
# Compare hybrid vs best classical and best DL on per-sample level
# Best classical: logistic regression (typically high performer)
best_cl_name = 'logistic_regression'
best_cl_idx  = base_model_names.index(best_cl_name) if best_cl_name in base_model_names else 0
pred_best_cl = (X_test_stack[:, best_cl_idx] >= 0.5).astype(int)

hybrid_correct  = (y_pred == y_test)
best_cl_correct = (pred_best_cl == y_test)

# Cases hybrid fixed
hybrid_corrected = (~best_cl_correct) & hybrid_correct
# Cases hybrid broke
hybrid_broke     = best_cl_correct & (~hybrid_correct)

print(f'Logistic Regression accuracy   : {best_cl_correct.mean():.4f}')
print(f'Hybrid accuracy                : {hybrid_correct.mean():.4f}')
print(f'Samples hybrid corrected       : {hybrid_corrected.sum():,}')
print(f'Samples hybrid broke           : {hybrid_broke.sum():,}')
print(f'Net improvement                : {hybrid_corrected.sum() - hybrid_broke.sum():+,}')

## 15. Save Hybrid Model

In [ ]:
payload = {
    'meta_learner'    : meta_lr,
    'base_model_names': base_model_names,
    'n_features'      : len(base_model_names),
}
out_path = os.path.join(HYBRID_DIR, 'stacking_lr.joblib')
joblib.dump(payload, out_path)
print(f'Hybrid model saved → {out_path}')

# Append to registry.json so the API picks it up
registry_path = os.path.join(MODELS_DIR, 'registry.json')
if os.path.exists(registry_path):
    with open(registry_path) as f:
        reg = json.load(f)
    reg['hybrid_stacking'] = {
        'display_name': MODEL_NAME,
        'category'    : 'Hybrid',
        'feature_type': 'hybrid',
        'path'        : os.path.relpath(out_path, BASE_DIR),
        'train_time'  : round(train_time, 2),
        'metrics'     : {
            'accuracy' : metrics['Accuracy'],
            'f1'       : metrics['F1 Score'],
            'precision': metrics['Precision'],
            'recall'   : metrics['Recall'],
            'auc'      : metrics['AUC-ROC'],
        },
    }
    with open(registry_path, 'w') as f:
        json.dump(reg, f, indent=2)
    print(f'Registry updated  → {registry_path}')
else:
    print('registry.json not found — run export_models.py first.')

## Summary

| Aspect | Detail |
|---|---|
| **Architecture** | Two-layer stacking: 22 base models → Logistic Regression meta-learner |
| **Base models** | 18 classical (TF-IDF) + 4 deep learning (LSTM/GRU/CNN/CNN-LSTM) |
| **Meta-features** | `prob_real` output from each base model (22-dim vector per sample) |
| **Meta-learner** | Logistic Regression (C=1.0, L2 regularisation) |
| **Training set** | 20% holdout from training data (base models not re-trained) |
| **Key advantage** | Combines complementary model signals; tolerates individual model errors |
| **Key insight** | Models with low mutual correlation contribute the most unique value |